In [2]:
import pandas as pd
import os
import numpy as np

In [3]:
output_dir = "results/processed_results_memory"
os.makedirs(output_dir, exist_ok = True)

In [4]:
memory_data = pd.read_csv("memory_usage_log.csv")

In [29]:
import pandas as pd
import numpy as np
import re

def memory_data_to_csv(memory_data, group_fun="mean"):
    expanded = memory_data["function_name"].str.split("__", expand=True)
    
    df = memory_data.copy()
    df["engine"] = expanded[0] + "_" + expanded[1]
    
    def transform_name(val):
        parts = val.split("_")
        if len(parts) != 2:
            return val
            
        first, second = parts[0], parts[1]
        
        if first.isdigit():
            first = f"{first}_nodes"
            
        if second == "SF":
            second = "scale_free"
        elif second.isdigit():
            second = f"{second}_edges"
            
        return f"{first}_{second}"

    df["name"] = expanded[2].apply(transform_name)
    df["operation"] = expanded[3]
    
    unique_engines = df["engine"].unique()
    dfs_by_engine = {}
    
    def natural_sort_key(s):
        return [int(text) if text.isdigit() else text.lower()
                for text in re.split('([0-9]+)', s)]

    for engine in unique_engines:
        subset = df[df["engine"] == engine]
        
        pivot_df = subset.pivot_table(
            index="name", 
            columns="operation", 
            values="peak_memory_mb",
            aggfunc=group_fun
        ).reset_index()
        
        pivot_df.columns.name = None
        
        pivot_df["sort_key"] = pivot_df["name"].apply(natural_sort_key)
        
        pivot_df = pivot_df.sort_values("sort_key", ascending =False).drop(columns=["sort_key"]).reset_index(drop=True)
        
        dfs_by_engine[engine] = pivot_df
        pivot_df.to_csv(f"{output_dir}/{engine}_{group_fun}.csv", index=True)
        
    return dfs_by_engine

results_std = memory_data_to_csv(memory_data, "std")
results = memory_data_to_csv(memory_data, "mean")

In [30]:
results["mysql_col"]

,name,create,delete,read_1,read_2,read_3,update_edges,update_nodes
0,ppi,0.011570,0.004739,4.361297,15.052898,29.044954,18.721001,0.693287
1,1015074_nodes_SBC,0.007839,0.004799,15.496849,16.171529,16.168975,60.715431,15.496887
2,1000000_nodes_scale_free,0.007835,0.004774,15.266838,1624.534832,3345.093483,30.786842,15.266998
3,1000000_nodes_5_edges,0.007833,0.004836,15.266838,57.987717,58.608787,76.301994,15.266865
4,100000_nodes_scale_free,0.007834,0.004723,2.170762,123.487486,332.387249,2.965492,1.533955
5,100000_nodes_20_edges,0.007833,0.004761,5.837167,24.156835,53.991285,30.525627,1.533992
6,100000_nodes_10_edges,0.007833,0.004745,2.953603,11.862811,14.590886,15.266899,1.533960
7,100000_nodes_5_edges,0.007832,0.004704,1.533928,5.910200,6.291286,7.637444,1.533953
8,10000_nodes_scale_free,0.007833,0.004742,13.056888,341.043061,482.483970,0.281502,0.160672
9,10000_nodes_20_edges,0.007832,0.004772,0.866717,3.153029,288.517125,3.059807,0.160685


In [21]:
results_std["mysql_col"]

,name,create,delete,read_1,read_2,read_3,update_edges,update_nodes
0,1000000_nodes_5_edges,0.000000,0.000250,0.000000,0.002038,0.001307,0.000000,0.000061
1,1000000_nodes_scale_free,0.000000,0.000000,0.000000,0.000654,0.000364,0.000000,0.000000
2,100000_nodes_10_edges,0.000000,0.000143,0.003504,0.001745,0.000962,0.000335,0.000065
3,100000_nodes_20_edges,0.000000,0.000140,0.002020,0.001051,0.003190,0.000000,0.000080
4,100000_nodes_5_edges,0.000000,0.000032,0.000000,0.001023,0.002950,0.000000,0.000058
5,100000_nodes_scale_free,0.000000,0.000106,0.000462,0.000096,0.000970,0.000000,0.000061
6,10000_nodes_10_edges,0.000000,0.000142,0.000405,0.001340,0.000536,0.000000,0.000061
7,10000_nodes_20_edges,0.000000,0.000168,0.003108,0.000426,0.005574,0.000000,0.000075
8,10000_nodes_5_edges,0.000000,0.000105,0.005978,0.001681,0.004071,0.000000,0.000074
9,10000_nodes_scale_free,0.000000,0.000154,0.000430,0.001056,0.001627,0.000000,0.000068


In [20]:
dfs_by_engine

{'mysql_col': operation      create    delete     read_1       read_2       read_3  \
 dataset                                                                
 1000000_5    0.007833  0.004836  15.266838    57.987717    58.608787   
 1000000_SF   0.007835  0.004774  15.266838  1624.534832  3345.093483   
 100000_10    0.007833  0.004745   2.953603    11.862811    14.590886   
 100000_20    0.007833  0.004761   5.837167    24.156835    53.991285   
 100000_5     0.007832  0.004704   1.533928     5.910200     6.291286   
 100000_SF    0.007834  0.004723   2.170762   123.487486   332.387249   
 10000_10     0.007832  0.004742   0.468529     1.426002    11.338828   
 10000_20     0.007832  0.004772   0.866717     3.153029   288.517125   
 10000_5      0.007831  0.004723   0.264545     0.764272     1.009423   
 10000_SF     0.007833  0.004742  13.056888   341.043061   482.483970   
 1000_10      0.007852  0.004742   0.115904     0.612469     5.024567   
 1000_20      0.007856  0.004707   0.1